In [ ]:
#from google.colab import drive
#import os
#drive.mount('/content/drive')
#path = '/content/drive/My Drive/Colab_Notebooks/NLP'
#os.chdir(path)

# Importing Libraries

**pip install only on colab**

In [ ]:
#pip install langchain-text-splitters

In [ ]:
#pip install bitsandbytes

In [ ]:
#pip install faiss-cpu

In [1]:
from datasets import load_dataset, get_dataset_config_names, Dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import numpy as np
from pathlib import Path
import re
import torch.nn.functional as F
from torch import Tensor
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss

# Inspecting Dataset

**Reference**:
_MTS, Department of War UAP Release 1 — structured corpus, 2026. CC-BY-4.0._
This dataset is a structured, machine-readable companion to the source material at [war.gov/UFO/](https://www.war.gov/UFO/).


<img src="https://media.mts-in.com/release_1/38-143685-box-incident-summaries-101-172/p137_f1_sketch.webp"
         alt="UFO"
         width="630"
         height="180">

In [2]:
ufo_dataset = "MTSLIVE/war-gov-uap-release-1"
get_dataset_config_names(ufo_dataset)   # dataset files

['documents', 'pages', 'figures', 'videos']

We will use the `pages` file.

In [3]:
pages = load_dataset(ufo_dataset, "pages", split="train")
pages

Dataset({
    features: ['document_id', 'page_no', 'text', 'has_figures'],
    num_rows: 4239
})

In [ ]:
# example of what we are going to use
print(pages[0]['text'])

## Check some documents by id

In [4]:
# all documents, each of these is composed of 1 or more pages
IDS = set(pages["document_id"])

Let's see if some pages have less than 20 characters.

In [ ]:
null_id = 0     # count for document_id (even if it's just one page)
null_pgs = 0    # count for pages (can share the same id)
doc_ids = []    # to filter later(?)

for page in pages:
    if len(page["text"]) <= 20:
        null_pgs+=1
        if page["document_id"] not in doc_ids:
            null_id+=1
            doc_ids.append(page["document_id"])
        #print(f"doc_id: {page["document_id"]}\n text: {page["text"]}", "\n")

print(f"total null ids (at least one page): {null_id}")
print(f"total null pages: {null_pgs}")
print(f"problematic ids: {doc_ids}")

So there are 61 `document_id` with _at least_ one page with less than 20 characters. If we talk in terms of pages, there are 225 pages almost empty.

In [5]:
count = pages.to_pandas().groupby("document_id").count()
single = list(count[count["page_no"]==1].index)  # Documents of 1 page only
# Print the single paged documents
# for doc in pages.filter(lambda x: x["document_id"] in single): print(f"DOCUMENT: {doc["document_id"].upper()}\n\n{doc["text"]}\n\n\n\n")

In [6]:
# Discard single paged documents containing no information
pattern = re.compile("fbi-photo|nasa-uap-vm|fbi-september-2023-sighting-composite-sketch")  # compile pattern to match discarded documents
useIDS = set(ID for ID in IDS if not pattern.match(ID))  # discard matches

In [7]:
filterPages = pages.filter(lambda x: x["document_id"] in useIDS and x["text"] != '')
# filterPages = pages.filter(lambda x: len(x["text"].strip()) <= 20)

# Embedding

The embedder is choosen from [here](https://huggingface.co/spaces/mteb/leaderboard)


**[Qwen3](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) Model Architecture**: <br>
is designed using dual-encoder and cross-encoder architectures the Embedding model processes a single text segment as input, extracting the semantic representation by utilizing the hidden state vector corresponding to the final [EOS] token. ([ref](https://qwen.ai/blog?id=qwen3-embedding))

<div style="display:flex; gap:20px;">
    <img src="https://miro.medium.com/v2/resize:fit:750/format:webp/1*jzZ_e5Bmvx84zPEa-LhqDQ.png"
         alt="Qwen3 architecture"
         width="330"
         height="180">
    <img src="https://miro.medium.com/v2/resize:fit:1400/format:webp/1*AWPQxx4xpiQGCp5XNNfyYA.png"
         alt="BERT vs Qwen3"
         width="350"
         height="180">
</div>

[article](https://arxiv.org/pdf/2506.05176)
For text embeddings, we utilize LLMs with causal attention, appending an
[EOS] token at the end of the input sequence. The final embedding is derived from the hidden state of the last layer corresponding to this [EOS] token.
To ensure embeddings follow instructions during downstream tasks, we concatenate the instruction and the query into a single input context, while leaving the document unchanged before processing with LLMs. The input format for queries is as follows:
`{Instruction}{Query}<|endoftext|>`

In [9]:
harrier_embedder = "microsoft/harrier-oss-v1-0.6b"
qwen_embedder = "Qwen/Qwen3-Embedding-4B"

In [8]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

In [11]:
# defining the embedding model and the relative tokenizer
# for the tokenizer we need the left padding

# harrier
tokenizer = AutoTokenizer.from_pretrained(harrier_embedder, padding_side='left', cache_dir='tokenizers_cache')
model = AutoModel.from_pretrained(harrier_embedder, quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir='models_cache')

# qwen
# tokenizer = AutoTokenizer.from_pretrained(qwen_embedder, padding_side='left', cache_dir='tokenizers_cache')
# model = AutoModel.from_pretrained(qwen_embedder, quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir="models_cache")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [12]:
# function to create the chunked text (input of embedder)
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer, chunk_size=512, chunk_overlap=77)

pages_chunked = []

for page in filterPages:
    page_text = page['text']
    chunk_list = text_splitter.split_text(page_text)
    for i, chunk in enumerate(chunk_list):
        pages_chunked.append({
            'document_id': page['document_id'],
            'page_no': page['page_no'],
            'chunk_id': i,
            'chunk_text': chunk
    })

chunkedPages = Dataset.from_list(pages_chunked)

In [13]:
# function to extract the last token (EOS) that is a compressed representation of the whole input (instruction+query)
def last_token_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

# we "merge" in one string instruction and query
def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery: {query}'

In [14]:
def get_embeddings(text_list, max_length=512):

  # Tokenize the input texts
  batch_dict = tokenizer(
    text_list,
    padding='longest',
    truncation=True,
    max_length=max_length,
    return_tensors="pt",
  )
  batch_dict.to(model.device)

  with torch.no_grad(): outputs = model(**batch_dict)
  embeddings = last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask']).cpu()

  torch.cuda.empty_cache()

  return embeddings

In [27]:
# docsEmbed = torch.load("harrier06DocsEmbed.pt", weights_only=False)  # colab
docsEmbed = torch.load("/mnt/eph/embedding/harrier06DocsEmbed.pt", weights_only=False)  # cv   #### kill eventually

# docsEmbed = torch.load("/mnt/eph/embedding/qwen4DocsEmbed.pt", weights_only=False)

In [28]:
embedPages = chunkedPages.add_column("embeddings", docsEmbed.to(torch.float32).cpu().numpy().tolist())

In [36]:
test = embedPages.to_pandas()

In [38]:
import pandas as pd

In [ ]:
pd.unique(test)

### Queries

Here we embed only the queries. We need to add the documents part
```python
# input_texts = chunks[:160]  # batchsize 80 for qwen3-0.6 40 for qwen3-4b  ## done in .py script
```

In [17]:
# Each query must come with a one-sentence instruction that describes the task
task = 'Given a document search query, retrieve relevant passages that answer the query'

raw_queries = [
    "What are Jesus's powers?",
    'What was spotted in the sky for the first time?',
    'Have UFOs ever been close to humans (astronauts)?',
    "What patterns emerge across the reported UAP sightings regarding location, altitude, behavior, speed, and time period?",
    "What are the most extravagant sightings?",
    "What is the Saucer's secret?",
    "What are they hiding from us?"
]

# ?queries = [get_detailed_instruct(task, query) for query in raw_queries]  ## qwen
queries = raw_queries  ## Harrier

In [18]:
queriesEmbed = get_embeddings(queries)

# Retrieval

In [19]:
# retrieving the top-k documents for each query
numpydocs = docsEmbed.to(torch.float32).cpu().numpy()
numpyqueries = queriesEmbed.to(torch.float32).cpu().numpy()
index = faiss.index_factory(docsEmbed.shape[1], "Flat", faiss.METRIC_INNER_PRODUCT)   # cosine similarity
faiss.normalize_L2(numpydocs)
index.add(numpydocs)
faiss.normalize_L2(numpyqueries)

k=5
k_best_distance, k_best_index = index.search(numpyqueries, k)

In [20]:
## Harrier
for i, query in enumerate(raw_queries):
    print(query)
    for j, idx in enumerate(k_best_index[i]):
        print(f"\nscore: {k_best_distance[i,j]}; idx: {idx}\n{chunkedPages[idx]["chunk_text"]}")
    print("\n\n\n")

What are Jesus's powers?

score: 0.6315035223960876; idx: 3304
The life of Jesus, now too, becomes clearer when these things are re-membered. His powers of levitation, His ability to pass through doors, walk on water, and heal the sick, are the essential attributes of men from outer space. They will also be ours someday when we will be "free like birds" (Ezk. 13:20). At Jesus' birth the celestial army came quite close to earth. A space-man appeared to the shepherds and the "glory" of the Lord, with the usual signs of His presence, shone around them. There was a multitude of the heavenly army with this space-man. And after their cosmic announcement, the music of their space-ships was heard as they again disappeared into space.

Jesus' ascension is described as "a cloud (or space-ship) received Him out of their sight" (Acts 1:9). His coming again is to be in the same manner. "Then will appear the sign of the Son of man in heaven (space) coming on the clouds (space-ships) of heaven (space

In [22]:
pages_chunked[1095]

{'document_id': '65-hs1-834228961-62-hq-83894-section-1',
 'page_no': 34,
 'chunk_id': 0,
 'chunk_text': 'FIRST ITS FLYING DISKS — NOW ITS "FIRE BALLS" PEOPLE SEE'}

In [75]:
## Qwen
for i, query in enumerate(raw_queries):
    print(query)
    for j, idx in enumerate(k_best_index[i]):
        print(f"\nscore: {k_best_distance[i,j]}\n{chunkedPages[idx]["chunk_text"]}")
    print("\n\n\n")

What are Jesus's powers?

score: 0.623723030090332
The life of Jesus, now too, becomes clearer when these things are re-membered. His powers of levitation, His ability to pass through doors, walk on water, and heal the sick, are the essential attributes of men from outer space. They will also be ours someday when we will be "free like birds" (Ezk. 13:20). At Jesus' birth the celestial army came quite close to earth. A space-man appeared to the shepherds and the "glory" of the Lord, with the usual signs of His presence, shone around them. There was a multitude of the heavenly army with this space-man. And after their cosmic announcement, the music of their space-ships was heard as they again disappeared into space.

Jesus' ascension is described as "a cloud (or space-ship) received Him out of their sight" (Acts 1:9). His coming again is to be in the same manner. "Then will appear the sign of the Son of man in heaven (space) coming on the clouds (space-ships) of heaven (space) with power

# Generation

In [52]:
def generation(input_tokenizer, input_model, messages, max_tokens):
  tokenizer = input_tokenizer
  model = input_model

  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
      enable_thinking=False   # try with True
  )
  model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

  # conduct text completion
  generated_ids = model.generate(
      **model_inputs,
      max_new_tokens=8192   # should be 32768?
  )
  output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

  # parsing thinking content
  try:
      # rindex finding 151668 (</think>)
      index = len(output_ids) - output_ids[::-1].index(151668)
  except ValueError:
      index = 0

  thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
  content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

  return thinking_content, content


In [53]:
qwen_generator = "Qwen/Qwen3-4B"

In [54]:
# qwen
tokenizer = AutoTokenizer.from_pretrained(qwen_generator, padding_side='left', cache_dir='tokenizers_cache')
model = AutoModelForCausalLM.from_pretrained(qwen_generator, quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir="models_cache")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:
# FROM DOCS https://huggingface.co/Qwen/Qwen3-4B more or less
system_prompt = "We are impartial detectives investigating the latest FBI UAP encounters. Use the provided documents to answer the Query as best as you can."
for i, query in enumerate(queries):
    ## through embedPages[int(idx)] you can access any metainformation of the document. It is not necessary possibly even useless to feed it to the generator
    retrDocs = [embedPages[int(idx)]["chunk_text"] for idx in k_best_index[i]]
    prompt = "Provided documents:\n"
    for j, doc in enumerate(retrDocs): prompt += f"Document {str(j + 1)}:\n{doc}.\n\n"    # here we can print metadata (doc_id and page_no) instead of whole text
    prompt += f"\nQuery: {query}"
    print(prompt)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    thinking_content, content = generation(tokenizer, model, messages, 16384)

    print("thinking content:", thinking_content)
    print("content:", content)

# Evaluation

Generators can be found [here](https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard#/)

In [11]:
test_pages = filterPages.shuffle(seed=32).select(range(15))

In [12]:
# da definire
test_generator = "Qwen/Qwen2.5-Coder-14B-Instruct"
#test_generator = "Qwen/Qwen3-4B-Instruct-2507"

In [13]:
# da definire
test_tokenizer = AutoTokenizer.from_pretrained(test_generator, padding_side='left', cache_dir='tokenizers_cache')
test_model = AutoModelForCausalLM.from_pretrained(test_generator, quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir="models_cache")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/47.5k [00:00<?, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [86]:
# two quastions/answers for each passage
system_prompt = """You are creating a benchmark for evaluating a RAG system.
Given a passage, generate exactly two question/answer pair.
Output format:

Question 1: ...
Gold Answer 1: ...
Question 2: ...
Gold Anser 2: ...
"""
qa_pairs = []
for num, passage in enumerate(test_pages):
    page_text = passage["text"]
    prompt = f"Passage:\n{page_text}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    thinking_content, content = generation(test_tokenizer, test_model, messages, 16384)    # max tokens to be decided
    qa_pairs.append(content)

    print(f"Passage {str(num + 1)}:", thinking_content)
    print("content:", content, "\n-------------------------")

Passage 1: 
content: Question 1: Who is Richard F. Shaver mentioned in the memorandum?
Gold Answer 1: Richard F. Shaver is a person who may have information concerning the origin of the "flying saucers" according to an unsigned telegram received by HQ. AAF.

Question 2: What was the purpose of the memorandum?
Gold Answer 2: The purpose of the memorandum was to request an investigation of Richard F. Shaver to determine whether he has information pertaining to the origin of Flying Saucers, based on the observation of flying saucers made by four witnesses in southern Wisconsin and their proximity to Lily Lake, Illinois. 
-------------------------
Passage 2: 
content: Question 1: Who moderated the colloquium on UFOs held at Pocantico in 1997?
Gold Answer 1: The colloquium was moderated by astrophysicist Peter Sturrock.

Question 2: What was the main focus of the colloquium organized by Laurance Rockefeller?
Gold Answer 2: The colloquium focused on physical evidence concerning UFOs. 
------

In [87]:
questions1 = re.findall(r'Question 1:\s*(.+?)(?=\n|$)', '\n'.join(qa_pairs))
gold_answers1 = re.findall(r'Gold Answer 1:\s*(.+)', '\n'.join(qa_pairs))
questions2 = re.findall(r'Question 2:\s*(.+?)(?=\n|$)', '\n'.join(qa_pairs))
gold_answers2 = re.findall(r'Gold Answer 2:\s*(.+)', '\n'.join(qa_pairs))

In [49]:
def save_file(new_file_name, obj_to_save):
  with open(new_file_name, 'w') as f:
    for line in obj_to_save:
        f.write(f"{line}\n")
  return print(f"File {new_file_name} saved")

def load_file(file_name):
    with open(file_name, "r") as f:
        obj_saved = [line.strip() for line in f]
    return obj_saved

In [88]:
save_file("questions1.txt", questions1)

File questions1.txt saved


In [89]:
save_file("questions2.txt", questions2)

File questions2.txt saved


In [90]:
save_file("gold_answers1.txt", gold_answers1)

File gold_answers1.txt saved


In [91]:
save_file("gold_answers2.txt", gold_answers2)

File gold_answers2.txt saved


Here we use our RAG to generate answer to the given test-questions.

In [92]:
# FROM DOCS https://huggingface.co/Qwen/Qwen3-4B more or less
system_prompt = """Given a question and the relative passage, generate the most accurate answer.
Output format:

Question 1: ...
Answer 1: ...
Question 2: ...
Answer 2: ...
"""
answers = []
for num, passage in enumerate(test_pages):
  prompt = f"Question 1: {questions1[num]}\nQuestion 2: {questions2[num]}\nPassage: {passage['text']}"
  messages = [
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": prompt}
  ]
  thinking_content, content = generation(tokenizer, model, messages, 16384)    # max tokens to be decided

  answers.append(content)

  print(f"Passage {str(num + 1)}:", thinking_content)
  print("content:", content, "\n-------------------------")

Passage 1: 
content: Question 1: Who is Richard F. Shaver mentioned in the memorandum?
Answer 1: Richard F. Shaver is a person in Lilly Lake, Illinois who may have information concerning the origin of the "flying saucers".

Question 2: What was the purpose of the memorandum?
Answer 2: The purpose of the memorandum was to request an investigation into whether Richard F. Shaver has information pertaining to the origin of flying saucers, based on the proximity of reported sightings to Lilly Lake, Illinois. 
-------------------------
Passage 2: 
content: Question 1: Who moderated the colloquium on UFOs held at Pocantico in 1997?
Answer 1: The colloquium was moderated by astrophysicist Peter Sturrock.

Question 2: What was the main focus of the colloquium organized by Laurance Rockefeller?
Answer 2: The main focus of the colloquium was to examine physical evidence concerning UFOs. 
-------------------------
Passage 3: 
content: Question 1: What was the purpose of Mr. Wacks' letter to Mr. Jo

In [93]:
rag_answers1 = re.findall(r'Answer 1:\s*(.+)', '\n'.join(answers))
rag_answers2 = re.findall(r'Answer 2:\s*(.+)', '\n'.join(answers))

In [97]:
save_file("rag_answers1.txt", rag_answers1)

File rag_answers1.txt saved


In [98]:
save_file("rag_answers2.txt", rag_answers2)

File rag_answers2.txt saved


Now that we have the questions, gold answers and answers we can use them as input for our test_generator that will act as a judge providing a score.

In [125]:
judge_input = {'quest': {'quest_1': questions1,
                         'quest_2': questions2},
               'gold_ans': {'quest_1': gold_answers1,
                            'quest_2': gold_answers2},
               'rag_ans': {'quest_1': rag_answers1,
                            'quest_2': rag_answers2}
               }

In [127]:
# to try
system_prompt = """You are a judge. Given a question, a gold answer and another second answer: you will give me a score for the second answer.
The score is a number between 0 and 100. The higher the score, the better the answer. The gold answer has a 100 score.
Output format for each passage X:
Passage X:
Question 1: ...
RAG answer score: .../100
Question 2: ...
RAG answer score: .../100
"""
scores = []
for num in range(len(test_pages)):
    
    quest1 = judge_input['quest']['quest_1'][num]
    quest2 = judge_input['quest']['quest_2'][num]
    goldansw1 = judge_input['gold_ans']['quest_1'][num]
    goldansw2 = judge_input['gold_ans']['quest_2'][num]
    ragansw1 = judge_input['rag_ans']['quest_1'][num]
    ragansw2 = judge_input['rag_ans']['quest_2'][num]

    prompt = f"Question 1: {quest1}\nGold answer 1: {goldansw1}\nRAG answer 1: {ragansw1}\n\nQuestion 2: {quest2}\nGold answer 2: {goldansw2}\nRAG answer 2: {ragansw2}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    thinking_content, content = generation(test_tokenizer, test_model, messages, 16384)

    scores.append(content)

    print(f"Passage {str(num + 1)}:", thinking_content)
    print("content:", content, "\n-------------------------")

Passage 1: 
content: Passage 1:
Question 1: Who is Richard F. Shaver mentioned in the memorandum?
RAG answer score: 85/100

Question 2: What was the purpose of the memorandum?
RAG answer score: 80/100 
-------------------------
Passage 2: 
content: Passage 1:
Question 1: Who moderated the colloquium on UFOs held at Pocantico in 1997?
RAG answer score: 100/100

Question 2: What was the main focus of the colloquium organized by Laurance Rockefeller?
RAG answer score: 100/100 
-------------------------
Passage 3: 
content: Passage 1:
Question 1: What was the purpose of Mr. Wacks' letter to Mr. Joseph F. Perry?
RAG answer score: 95/100

Question 2: To whom should Mr. Perry direct further inquiries about the return of the photographs?
RAG answer score: 98/100 
-------------------------
Passage 4: 
content: Passage 1:
Question 1: Who is the commanding general addressed in the passage?
RAG answer score: 100/100

Question 2: What is the subject of the forwarded information?
RAG answer score: 7

In [133]:
scores[:2]

['Passage 1:\nQuestion 1: Who is Richard F. Shaver mentioned in the memorandum?\nRAG answer score: 85/100\n\nQuestion 2: What was the purpose of the memorandum?\nRAG answer score: 80/100',
 'Passage 1:\nQuestion 1: Who moderated the colloquium on UFOs held at Pocantico in 1997?\nRAG answer score: 100/100\n\nQuestion 2: What was the main focus of the colloquium organized by Laurance Rockefeller?\nRAG answer score: 100/100']

In [134]:
for passage in scores:
    scores1 = re.findall(r'Question (\d+):.*?RAG answer score: (\d+)/100', scores, re.DOTALL)

TypeError: expected string or bytes-like object, got 'list'

In [131]:
scores1

['85/100',
 '80/100',
 '100/100',
 '100/100',
 '95/100',
 '98/100',
 '100/100',
 '70/100',
 '100/100',
 '100/100',
 '100/100',
 '0/100',
 '85/100',
 '90/100',
 '85/100',
 '90/100',
 '100/100',
 '85/100',
 '95/100',
 '90/100',
 '95/100',
 '98/100',
 '100/100',
 '100/100',
 '95/100',
 '98/100',
 '99/100',
 '99/100',
 '85/100',
 '90/100']